# Prétraitement du dataset Diabète – Sprint 1

## Contexte du projet (énoncé officiel)
Le projet vise la **prédiction du diabète** à partir des données BRFSS 2015 (CDC/Kaggle), avec un enjeu de santé publique fort : améliorer le repérage précoce des patients à risque.

Données de départ (énoncé) :
- 253 680 répondants,
- 22 variables,
- objectif final de modélisation : classification binaire, évaluée principalement en **ROC AUC**.

## Objectif de ce notebook
Ce notebook couvre la partie **Sprint 1 – Prétraitement** pour produire un jeu de données propre, traçable et réutilisable par les sprints de modélisation.

Pipeline implémenté ici :
1. Lecture et inspection du dataset source,
2. Séparation cible / variables explicatives,
3. Découpage train / validation / test,
4. Normalisation des variables,
5. Sauvegarde des artefacts.

## Livrables produits
- `data/processed/train.parquet`
- `data/processed/val.parquet`
- `data/processed/test.parquet`
- `artifacts/scaler.joblib`

Ces sorties permettent de lancer l’entraînement de manière reproductible et cohérente avec les objectifs de l’énoncé.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## Lecture du dataset

Cette étape vérifie la qualité des données d’entrée avant toute transformation.

Nous affichons :
- un aperçu des premières lignes (contrôle rapide du format),
- les types de colonnes et le nombre total d’observations,
- le nombre de valeurs manquantes par variable.

Pourquoi c’est important : un diagnostic initial explicite évite de propager des erreurs silencieuses dans tout le pipeline (types incohérents, colonnes inattendues, données incomplètes).

In [ ]:
def read_data(file_path="../data/raw/diabetes_binary_health_indicators_BRFSS2015.csv"):
    try:
        df = pd.read_csv(file_path)
        print("Aperçu des 5 premières lignes :")
        print(df.head())
        print("\nInformations générales :")
        print(df.info())
        print("\nValeurs manquantes par colonne :")
        print(df.isnull().sum())
        df.to_parquet("test_index_false.parquet", index=False)
        return df
    except FileNotFoundError:
        print(f"Erreur : le fichier {file_path} n'a pas été trouvé.")
        return None

df = read_data()

## Prétraitement initial

Selon l’énoncé, la cible d’origine peut être codée en 3 classes (`0`, `1`, `2`) et doit être recodée en binaire :
- `0` et `1` -> `0` (**No diabetes / Prediabetes**),
- `2` -> `1` (**Diabetes**).

Dans cette implémentation, nous utilisons la colonne `Diabetes_binary` déjà au format binaire dans le dataset chargé, ce qui est cohérent avec la variante Kaggle utilisée.

Nous séparons ensuite :
- `X` : les variables explicatives,
- `y` : la cible.

Ce découpage isole clairement les entrées du modèle (`X`) de la variable à prédire (`y`) et prépare les étapes d’entraînement/évaluation.

In [ ]:
def separate_data(df):
    X = df.drop("Diabetes_binary", axis=1)
    y = df["Diabetes_binary"]
    return X, y

if df is not None:
    X, y = separate_data(df)
    print("Distribution des classes :")
    print(y.value_counts())

## Séparation Train / Validation / Test

Répartition appliquée :
- 70% entraînement,
- 15% validation,
- 15% test.

La séparation est faite avec **stratification** sur la cible pour conserver la proportion des classes dans chaque sous-ensemble.

Rôle de chaque split :
- **Train** : apprentissage des paramètres du modèle,
- **Validation** : réglage des hyperparamètres et choix de modèle,
- **Test** : estimation finale des performances sur données non vues.

Cette organisation limite les biais d’évaluation et améliore la robustesse des conclusions.

In [ ]:
if df is not None:
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)
    
    print("Taille train :", X_train.shape)
    print("Taille validation :", X_val.shape)
    print("Taille test :", X_test.shape)
    print(f"nb de ligne : {len(df)}")

## Normalisation des features

Nous appliquons une standardisation (`StandardScaler`) pour mettre les variables sur une échelle comparable :
- moyenne proche de 0,
- écart-type proche de 1.

Bonne pratique suivie :
- `fit` uniquement sur le jeu d’entraînement,
- `transform` sur validation et test.

Ce choix évite la fuite de données (data leakage) et garantit une évaluation réaliste. Il facilite aussi la convergence de nombreux algorithmes de machine learning/deep learning.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Normalisation effectuée. Exemple :")
print(X_train_scaled[:5])

## Sauvegarde des datasets prétraités

Pour rendre le travail **livrable et reproductible**, nous exportons les jeux de données normalisés ainsi que le scaler appris.

Organisation des dossiers :
- `data/raw` : source brute,
- `data/processed` : jeux prêts pour l’entraînement,
- `artifacts` : objets techniques réutilisables (ici le scaler).

Éléments sauvegardés :
- `train.parquet`,
- `val.parquet`,
- `test.parquet`,
- `scaler.joblib`.

Cette étape permet à toute l’équipe de reprendre le projet sans refaire le prétraitement, avec les mêmes transformations et donc des résultats comparables.

In [ ]:
import os
import pandas as pd
import joblib

# Création des dossiers projet
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)
os.makedirs("artifacts", exist_ok=True)

# Reconstruction des DataFrames à partir des données normalisées
df_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
df_train_scaled["Diabetes_binary"] = y_train.reset_index(drop=True)

df_val_scaled = pd.DataFrame(X_val_scaled, columns=X_val.columns)
df_val_scaled["Diabetes_binary"] = y_val.reset_index(drop=True)

df_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)
df_test_scaled["Diabetes_binary"] = y_test.reset_index(drop=True)

# Sauvegarde des datasets
df_train_scaled.to_parquet("data/processed/train.parquet", index=False)
df_val_scaled.to_parquet("data/processed/val.parquet", index=False)
df_test_scaled.to_parquet("data/processed/test.parquet", index=False)

# Sauvegarde du scaler
joblib.dump(scaler, "artifacts/scaler.joblib")

print("Datasets sauvegardés dans data/processed/")
print("Scaler sauvegardé dans artifacts/")

## Alignement Sprint 1 : réalisé, limites, suite

### Tâches Sprint 1 réalisées dans ce notebook
- Chargement et compréhension initiale du dataset,
- Séparation cible / features,
- Scission train / validation / test (stratifiée),
- Normalisation des variables,
- Sauvegarde des jeux prétraités et du scaler.

### Points partiellement couverts ou à compléter
- Typage fin des variables (catégorielles vs numériques) à expliciter davantage,
- Nettoyage avancé des doublons et stratégie détaillée de gestion des valeurs manquantes,
- EDA quantitative et visuelle plus complète.

### Hypothèses
- Le fichier BRFSS 2015 utilisé correspond bien à la version attendue,
- La cible `Diabetes_binary` est conforme au recodage demandé par l’énoncé.

### Critères d’acceptation du livrable actuel
- `train.parquet`, `val.parquet`, `test.parquet` générés,
- `artifacts/scaler.joblib` généré,
- séparation reproductible (`random_state=42`) et stratifiée,
- documentation textuelle explicite des choix méthodologiques.

Ce livrable fournit une base opérationnelle pour le Sprint 2 (construction et évaluation du premier réseau de neurones).